In [1]:
from swarmbots.mj_env.scenarios.obstacle_street_scenario import ObstacleStreetScenario
from swarmbots.mj_env.swarm.simple_swarm_tetrahedron_zx import SimpleSwarmTetrahedronZX
from swarmbots.mj_env.swarm_bots_env import SwarmBotsEnv
%load_ext autoreload
%autoreload 2

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize, DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import imageio


In [3]:


class NoConnectionActionWrapper(gym.Wrapper):
    """
    Wraps a SwarmBotsEnv to flatten the observation space and
    convert the hybrid Dict action space into a single flat Box space.
    """
    def __init__(self, env: SwarmBotsEnv):
        super().__init__(env)
        self.env = env

        # --- 1. Flatten Observation Space ---
        # Original obs is (n_agents, obs_per_agent) or similar
        # We flatten it to (n_agents * obs_per_agent,)
        self.obs_shape = env.observation_space.shape
        self.obs_dim_flat = np.prod(self.obs_shape)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.obs_dim_flat,),
            dtype=np.float32
        )

        # --- 2. Flatten Action Space (Hybrid -> Box) ---
        # The mj_env action space is a Dict with 'actuators' (Box) and 'connectors' (MultiBinary)
        self.original_action_space = env.action_space

        # Get shapes
        self.act_space = self.original_action_space['actuators']

        self.act_shape = self.act_space.shape

        self.act_dim_flat = np.prod(self.act_shape)

        # Create a single continuous action space for ppo
        # We use [-1, 1] range. For binary actions, >0 will be treated as 1.
        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(self.act_dim_flat,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        return obs.flatten(), info

    def step(self, action):
        # 'action' is a flat float array from ppo

        # 1. Split into actuator and connector parts
        act_part_flat = action[:self.act_dim_flat]

        # 2. Reshape actuators (Continuous)
        # ppo outputs in [-1, 1] which matches the actuator space usually
        actuators = act_part_flat.reshape(self.act_shape)

        # 4. Construct Dictionary Action
        dict_action = {
            'actuators': actuators,
            'connectors': self.env.swarm_connections.get_is_active_mask()
        }

        obs, reward, terminated, truncated, info = self.env.step(dict_action)

        return obs.flatten(), reward, terminated, truncated, info

def make_env():
    rng = np.random.default_rng(42)
    swarm = SimpleSwarmTetrahedronZX(connection_torquescale=10.0)
    scenario = ObstacleStreetScenario(swarm=swarm, payload_type=None, seed=rng.integers(0, 10000000))
    env = SwarmBotsEnv(scenario=scenario, render_mode=None)
    # Use the new wrapper
    env = NoConnectionActionWrapper(env)
    env = Monitor(env)
    print(f'{env.observation_space = }')
    print(f'{env.action_space = }')
    return env

def record(model_path="ppo_no_con_swarm_bots", vecnorm_path="vecnormalize_swarm_bots.pkl"):
    def make_eval_env():
        swarm = SimpleSwarmTetrahedronZX(connection_torquescale=10.0)
        scenario = ObstacleStreetScenario(swarm=swarm, payload_type=None, seed=42)
        env = SwarmBotsEnv(scenario=scenario, render_mode="rgb_array", width=640, height=480)
        env = NoConnectionActionWrapper(env)
        return env

    model = PPO.load(model_path)
    eval_env = DummyVecEnv([make_eval_env])
    eval_env = VecNormalize.load(vecnorm_path, eval_env)
    eval_env.training = False
    eval_env.norm_reward = False

    obs = eval_env.reset()
    images = []
    print("Recording rollout...")

    for _ in range(500):
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)
        img = eval_env.envs[0].render()
        if img is not None:
            images.append(img)
        if dones[0]:
            break

    eval_env.close()

    if images:
        imageio.mimsave("swarm_bots_rollout_ppo_no_con.gif", images, fps=30)
        print("Saved swarm_bots_rollout_ppo_no_con.gif")
    else:
        print("No images captured.")

vec_env = SubprocVecEnv([make_env] * 4)
print(vec_env.action_space)
print(vec_env.observation_space)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True)

policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))
model = PPO(
    "MlpPolicy",
    vec_env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    n_steps=512,
    device="cpu",
    learning_rate=2e-5,
    target_kl=0.05
)

print("Starting training...")
model.learn(total_timesteps=10_000_000)
print("Training finished.")

model.save("ppo_no_con_swarm_bots")
vec_env.save("vecnormalize_swarm_bots.pkl")

record()


In [6]:

record()